# 실습 B. AI로 Dafny 모델 만들기 (약 40분)

Java 코드는 그대로 두고, 그 옆에 **같은 일을 하는 Dafny 코드**(그림자 모델)를 둡니다. Dafny는 각 메서드에 적힌 계약(지켜야 할 약속)을 코드가 정말 지키는지 **모든 입력에 대해** 증명합니다.

| 문법 | 뜻 |
|---|---|
| `requires P` | 이 메서드를 부르기 전에 P가 참이어야 한다 (사전조건) |
| `ensures Q` | 이 메서드가 끝나면 Q가 참이다 (사후조건) |
| `old(x)` | 메서드가 시작될 때의 x 값 |
| `ghost var elems: seq<Event>` | 증명에만 쓰는 변수. 실행되지 않습니다. 여기서는 "대기열에 든 이벤트 목록(오래된 것부터)" |
| `modifies x` | 이 메서드가 바꿀 수 있는 것 |
| `A ==> B` / `A <==> B` | A이면 B / A와 B는 같이 참이거나 같이 거짓 |

증명이 통과해도 확인되는 것은 "Dafny 모델이 계약을 지킨다"까지입니다. 그래서 두 가지를 따로 확인합니다.
1. **계약이 맞게 쓰였는가**: 사람이 읽어서 판단합니다. 계약이 약하면 틀린 코드도 증명을 통과합니다.
2. **모델이 Java와 같게 동작하는가**: 차분 테스트로 확인합니다.

In [1]:
import lab   # 실습 도우미: lab.run(...)이 make 명령을 실행하고 결과 보고서를 아래에 보여 줍니다

## B-1. AI가 만든 모델 읽기 (15분)

AI에게 아래 한 문장과 `RingEventQueue.java`만 주고 Dafny 모델을 만들게 했습니다(프롬프트 P-1). 일부러 아주 짧게 물었습니다.

> 다음 Java 클래스를 Dafny로 만들어 줘. dafny verify가 통과해야 해.

수업 중에 같은 일을 시키면 AI가 한 번에 검증을 통과하는 모델을 내지 못하는 경우가 많고 시간도 듭니다. 그래서 **미리 받아 둔 결과 예시**(`scenarios/b1-ai-sample/`)로 진행합니다. 이 모델을 믿어도 되는지 두 단계로 확인합니다.

**1단계: 검증과 차분 테스트.** 예시 모델 `RingEventQueue.dfy`와 연결 코드 `MyQueueAdapter.java`(Dafny가 만든 Java 코드를 차분 테스트 도구에 이어 주는 코드)로 검증 → Java로 컴파일 → 차분 테스트를 한 번에 돌립니다.

In [6]:
lab.run("difftest-sample")

$ make difftest-sample


Dafny RingEventQueue.dfy                       검증 19개, 오류 0개


연결 코드 컴파일: 성공


[queue] 300 runs, 18000 ops: Java and model agree


Dafny 검증 오류 0개, 차분 테스트(무작위 연산열 300개)도 모두 일치합니다. 하지만 **계약이 약하면 틀린 모델도 검증을 통과합니다.** 그래서 2단계로 계약을 읽습니다.

**2단계: 계약 읽기.**

예시 결과 `scenarios/b1-ai-sample/RingEventQueue.dfy`를 처음부터 끝까지 읽어 봅니다. 줄마다 색으로 묶고 오른쪽에 뜻을 적었습니다. 색의 뜻은 이렇습니다.

| 색 | 무엇 | 읽을 때 |
|---|---|---|
| 파랑 | Java 필드를 옮긴 것 | Java와 같은지 봅니다 |
| 보라 | 증명용 변수(`ghost`) | 실행되지 않습니다. 계약은 이것으로 말합니다 |
| 초록 | 불변식 `Valid()` | 실행되는 값과 증명용 변수가 맞는다는 약속 |
| 주황 / 분홍 / 청록 | 사전조건 `requires` / 바꿀 수 있는 것 `modifies` / 사후조건 `ensures` | **계약. 사람이 꼭 읽고 판단할 부분** |
| 회색 | 본문, 함수 | Java 코드를 옮긴 부분 |
| 흐림 | 증명 도우미 | 계약이 아니라 Dafny가 증명을 찾도록 돕는 계산 과정. 건너뛰어도 됩니다 |

**① 상태: 필드, 증명용 변수, 불변식, 생성자**

In [7]:
lab.annotate("scenarios/b1-ai-sample/RingEventQueue.dfy", [
    (13, 13, "body", "Java 클래스 <code>RingEventQueue</code>를 옮긴 Dafny 클래스"),
    (14, 16, "java", "Java 필드 <code>buf</code>, <code>head</code>, <code>count</code>를 그대로 옮긴 것. 실행되는 코드입니다"),
    (17, 17, "ghost", "<b>대기열에 든 이벤트 목록, 오래된 것부터.</b> 실행되지 않고 증명에만 씁니다. 아래 계약은 모두 배열 대신 이 목록으로 말합니다(맨 뒤에 추가된다, 맨 앞이 빠진다)"),
    (19, 21, "inv", "<code>Valid()</code>: 배열과 <code>elems</code>가 서로 맞는다는 약속(불변식). 모든 메서드가 시작할 때 이것을 믿고, 끝날 때 다시 지켜야 합니다"),
    (22, 25, "inv", "칸이 1개 이상, head는 배열 안, count는 0~칸 수, <code>elems</code>의 길이 = count"),
    (26, 27, "inv", "<b>핵심:</b> <code>elems</code>의 i번째 = 배열의 <code>(head + i) % 칸 수</code>번째 칸. 끝을 넘으면 앞으로 돌아갑니다(아래 그림)"),
    (29, 29, "body", "생성자 = Java <code>RingEventQueue(int capacity)</code>"),
    (30, 30, "pre", "<code>capacity &gt; 0</code>일 때만 부를 수 있음. Java는 이 경우 예외를 던집니다"),
    (31, 32, "post", "끝나면 <code>Valid()</code>가 참이고, <code>elems</code>는 빈 목록, 칸 수 = capacity. <code>fresh(buf)</code>: 배열은 새로 만든 것"),
    (33, 38, "body", "Java 생성자를 옮긴 본문. 37줄은 증명용 변수의 시작값"),
], start=13, end=38, title="AI 결과 예시 RingEventQueue.dfy: 상태")

26줄을 그림으로 보면 이렇습니다. 칸 4개짜리 배열에 c, d, e, f가 들었고 head가 2인 경우입니다. 배열에서는 e, f가 앞쪽에 있지만, `elems`는 head부터 읽어 **c, d, e, f** 순서입니다.

In [8]:
lab.queue_picture(["e", "f", "c", "d"], head=2, count=4, title="Valid() 26줄: elems[i] == buf[(head + i) % buf.Length]")

**② Offer: 계약 한 줄씩**

In [9]:
lab.annotate("scenarios/b1-ai-sample/RingEventQueue.dfy", [
    (40, 40, "body", "Java <code>offer(Event e)</code>. 결과 <code>r</code>은 OK / FULL / REJECTED"),
    (41, 41, "pre", "<b>부르기 전 약속:</b> 대기열이 <code>Valid()</code> 상태여야 함. 이것 말고는 조건이 없으므로 가득 찼을 때도 부를 수 있습니다"),
    (42, 42, "mod", "<b>바꿀 수 있는 것:</b> 이 객체의 필드(<code>this</code>)와 배열의 내용(<code>buf</code>). 다른 것은 바꾸지 않음"),
    (43, 43, "post", "끝나면 다시 <code>Valid()</code>, 배열은 같은 배열(<code>buf == old(buf)</code>: 새 배열로 바꾸지 않음). <code>old(x)</code>는 시작할 때의 x"),
    (44, 44, "post", "<b>자리가 있었으면</b>(시작할 때 count &lt; 칸 수): 결과는 OK이고, <code>elems</code>는 <b>맨 뒤에 e가 붙은 것</b>"),
    (45, 45, "post", "<b>가득 찼으면</b>: 결과는 FULL이고, <code>elems</code>는 <b>그대로</b>"),
    (46, 52, "body", "Java 32~34줄을 옮긴 본문. 52줄은 증명용 변수도 같이 바꿈"),
    (53, 58, "proof", "모든 i에 대해 26줄이 여전히 맞다는 것을 Dafny가 확인하도록 돕는 부분. 보조정리 <code>ModLemma</code>는 % 계산에 대한 사실"),
    (59, 60, "body", ""),
], start=40, end=60, title="AI 결과 예시 RingEventQueue.dfy: Offer")

44·45줄이 Offer의 뜻 전부입니다. 두 줄이 **모든 경우**(자리 있음 / 가득 참)를 나누고, 각각 **결과값**과 **바뀐 뒤의 내용**을 함께 말합니다.

**③ Poll, Peek: 같은 방식으로**

In [10]:
lab.annotate("scenarios/b1-ai-sample/RingEventQueue.dfy", [
    (62, 62, "body", "Java <code>poll()</code>. Java는 비었으면 <code>null</code>을 돌려주는데, 이 모델은 <code>(e, ok)</code> 두 값을 돌려주고 비었으면 <code>ok = false</code>"),
    (63, 64, "pre", "Offer와 같음: <code>Valid()</code>이면 부를 수 있고, 필드와 배열 내용을 바꿀 수 있음"),
    (65, 65, "post", "끝나면 다시 <code>Valid()</code>, 같은 배열"),
    (66, 66, "post", "<b>비어 있었으면</b>: ok = false, <code>elems</code> 그대로"),
    (67, 67, "post", "<b>들어 있었으면</b>: ok = true, e = 가장 오래된 것(<code>old(elems)[0]</code>), <code>elems</code>는 맨 앞을 뺀 나머지(<code>[1..]</code>)"),
    (68, 78, "body", "Java 39~44줄을 옮긴 본문"),
    (79, 85, "proof", "26줄이 여전히 맞다는 증명 돕기"),
    (86, 87, "body", ""),
    (89, 89, "body", "Java <code>peek()</code>"),
    (90, 90, "pre", "<code>modifies</code> 줄이 없음 = <b>아무것도 바꾸지 않음</b>"),
    (91, 92, "post", "들어 있는지(ok), 들어 있으면 e = 맨 앞"),
    (93, 99, "body", ""),
], start=62, end=99, title="AI 결과 예시 RingEventQueue.dfy: Poll, Peek")

**④ 나머지: 함수, Clear, Snapshot, 보조정리**

In [11]:
lab.annotate("scenarios/b1-ai-sample/RingEventQueue.dfy", [
    (101, 117, "func", "<code>function</code>: 값을 계산하기만 하는 것. 계약 대신 본문 자체가 뜻입니다(Size = count, Capacity = 칸 수, IsEmpty = count가 0)"),
    (119, 119, "body", "Java <code>clear()</code>"),
    (120, 122, "pre", "Valid()이면 부를 수 있고, 필드와 배열 내용을 바꿀 수 있음. 끝나면 Valid(), 같은 배열"),
    (123, 123, "post", "<b>끝나면 비어 있음</b>"),
    (124, 133, "body", "Java처럼 Poll을 반복. <code>invariant</code>(반복할 때마다 참인 것)와 <code>decreases</code>(반복이 끝난다는 근거)는 증명 도우미"),
    (135, 135, "body", "Java <code>snapshot()</code>"),
    (136, 137, "post", "<b>돌려주는 목록 = elems 그대로</b>"),
    (138, 148, "body", "Java 72~74줄을 옮긴 반복"),
    (151, 173, "proof", "보조정리(lemma): % 계산에 대한 사실을 증명해 두고 위에서 가져다 씀. 계약이 아님"),
], start=101, end=173, title="AI 결과 예시 RingEventQueue.dfy: 나머지")

### Java와 나란히 놓기

어느 Java 줄이 어느 Dafny 줄이 됐는지 봅니다. **옮기면서 달라진 곳**(오른쪽 설명의 굵은 글씨)이 판단할 거리입니다.

In [12]:
lab.side_by_side([
    (("legacy/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java", 15, 17), ("scenarios/b1-ai-sample/RingEventQueue.dfy", 14, 17), "필드 세 개는 그대로. Dafny에는 증명용 <code>elems</code>가 더 있음"),
    (("legacy/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java", 30, 36), ("scenarios/b1-ai-sample/RingEventQueue.dfy", 40, 52), "가득 참이면 FULL, 아니면 <code>(head + count) % 칸 수</code>에 넣고 count 증가: 같음. <b>Java 31줄 <code>null</code> → REJECTED는 모델에 없음.</b> 모델의 <code>Event</code>는 null이 될 수 없는 타입이라 이 경우를 다루지 않습니다"),
    (("legacy/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java", 38, 45), ("scenarios/b1-ai-sample/RingEventQueue.dfy", 62, 78), "꺼낸 칸을 비우고 head를 한 칸 옮김: 같음. <b>빈 칸 표시가 다름</b>: Java는 <code>null</code>, 모델은 <code>Event(0, false)</code>. 비었을 때의 결과도 Java <code>null</code>, 모델 <code>ok = false</code>"),
    (("legacy/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java", 64, 68), ("scenarios/b1-ai-sample/RingEventQueue.dfy", 119, 133), "Poll을 빌 때까지 반복: 같음"),
    (("legacy/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java", 70, 76), ("scenarios/b1-ai-sample/RingEventQueue.dfy", 135, 148), "head부터 count개를 차례로: 같음"),
], title="legacy RingEventQueue.java ↔ AI 결과 예시 RingEventQueue.dfy")

### 계약 확인 목록

AI가 만든 모델의 public 메서드마다 아래를 확인합니다. 오른쪽은 위 예시에서 찾은 답입니다.

| 확인할 것 | 빠지면 생기는 일 | 예시에서 |
|---|---|---|
| 모든 경우를 다루는가? (자리 있음 / 가득 참, 비어 있음 / 비어 있지 않음) | 다루지 않은 경우에는 아무 동작이나 증명을 통과합니다 | Offer 44·45줄, Poll 66·67줄: 두 경우 모두 있음 |
| 결과값뿐 아니라 **바뀐 뒤의 내용**까지 말하는가? (`elems == ...`) | FULL을 돌려주면서 내용을 망가뜨려도 통과합니다 | 44줄 `elems == old(elems) + [e]`, 67줄 `elems == old(elems)[1..]` |
| 바뀌면 안 되는 것이 그대로라고 말하는가? (`elems == old(elems)`) | 위와 같음 | 45줄, 66줄. 43·65줄 `buf == old(buf)` |
| `requires`가 너무 강하지 않은가? | Java에서는 가능한 호출을 모델이 다루지 않게 됩니다 | Offer는 `Valid()`만 요구(41줄). 만약 "가득 차 있지 않을 때만"을 요구했다면 45줄 경우를 다루지 않는 모델이 됩니다 |
| 모델 밖으로 둔 동작이 적혀 있는가? (null 입력, 동시성 등) | 무엇을 확인하지 않았는지 모르게 됩니다 | **빠져 있음.** Java 31줄 `null` → REJECTED를 다루지 않는데, 파일 어디에도 적혀 있지 않습니다. AI에게 고치라고 하거나 주석으로 남겨야 합니다 |

### (선택) 직접 AI에게 시켜 보기

시간이 남으면 위 과정을 직접 해 봅니다. 아래 셀을 차례로 실행하면 Codex가 만든 모델과 연결 코드가 `mine/`에 저장되고, 검증과 차분 테스트를 돌립니다. 이 환경에서 해 봤을 때는 AI 모델이 한 번에 검증을 통과하지 못했습니다(Dafny 오류 4개). 이것도 "AI가 만든 것은 검증으로 판단한다"의 예입니다.

검증이 실패하면 마지막 셀이 오류를 프롬프트 P-2("계약 줄은 한 글자도 바꾸지 마라")에 넣어 고치게 합니다. 고치기 전 파일과 나란히 보여 주니 **requires / ensures 줄이 바뀌지 않았는지** 확인하고, `difftest-mine` 셀을 다시 실행합니다.

In [ ]:
lab.codex("P-1", save="mine/RingEventQueue.dfy")        # 모델 만들기

In [ ]:
lab.codex("P-1b", save="mine/MyQueueAdapter.java")      # 연결 코드 만들기

In [ ]:
lab.run("difftest-mine")                                # 검증 → 컴파일 → 차분 테스트

In [ ]:
# 검증이 실패했을 때만: 오류를 주고 고치게 함(계약 줄은 그대로 두라고 지시)
lab.codex("P-2", 파일="mine/RingEventQueue.dfy", 오류=lab.dafny_errors("mine/RingEventQueue.dfy"), save="mine/RingEventQueue.dfy")

## B-2. 기준 모델 검증 (5분)

B-2부터는 강사가 미리 만들어 검증해 둔 **기준 모델**(`formal/` 폴더: 대기열, Dispatcher, 전력)을 씁니다. Java 메서드와 Dafny 메서드의 대응은 `formal/mapping.tsv`에 있습니다.

기준 대기열 모델 `formal/RingQueue.dfy`는 B-1의 예시와 같은 구조이고 이름만 다릅니다.

| | B-1 AI 결과 예시 | 기준 모델 |
|---|---|---|
| 파일 | `scenarios/b1-ai-sample/RingEventQueue.dfy` | `formal/RingQueue.dfy` |
| 클래스 | `RingEventQueue` | `RingQueue` |
| 증명용 변수(대기열 내용) | `elems` | `Contents` |
| 칸 수 | `buf.Length` | `Capacity()` |
| 모델 밖으로 둔 동작 | 적혀 있지 않음 | 파일 맨 위 8줄에 적혀 있음(`offer(null)`, 스레드 등) |

In [13]:
lab.run("verify")

$ make verify


Dafny dfyconfig.toml                           검증 71개, 오류 0개


검증 오류 0개입니다. 기준 모델의 모든 메서드가 각자 적힌 계약을 지킨다는 뜻입니다.

## B-3. 계약 읽기 연습 (10분)

HomeHub에는 대기열이 가득 찼을 때 바꾸면 안 되는 규칙 R1~R4가 있습니다.

| 번호 | 규칙 |
|---|---|
| R1 | 대기 이벤트 수는 정해진 상한을 넘지 않는다 |
| R2 | 누수·과열 같은 중요(critical) 이벤트는 절대 거부하지 않는다 |
| R3 | 대기열에 남는 이벤트의 순서는 바뀌지 않는다. 자리를 만들어야 하면 가장 오래된 비중요 이벤트를 밀어낸다 |
| R4 | 비중요 이벤트는 대기열이 가득 차 있고 autoDrain 설정이 꺼져 있을 때만 거부한다 |

Dafny로 "코드가 R1~R4를 지키는가"를 판단하려면, 이 규칙이 계약의 어느 줄에 적혀 있는지 읽을 수 있어야 합니다. 두 계약을 봅니다. 둘 다 기준 모델(`formal/`)입니다.

**① 대기열 `RingQueue.Offer`의 계약.** B-1에서 읽은 예시의 Offer 44·45줄과 같은 내용이고, 이름만 `Contents`·`Capacity()`입니다.

In [14]:
lab.annotate("formal/RingQueue.dfy", [
    (70, 70, "body", "Java <code>offer</code>의 기준 모델"),
    (71, 71, "pre", "Valid() 상태면 부를 수 있음"),
    (72, 72, "mod", "필드와 배열 내용을 바꿀 수 있음"),
    (73, 73, "post", "끝나면 Valid(), 같은 배열"),
    (74, 74, "post", "<b>자리가 있었으면</b>: OK, 내용 맨 뒤에 e 추가"),
    (75, 75, "post", "<b>가득 찼으면</b>: FULL, 내용 그대로"),
], start=70, end=75, title="formal/RingQueue.dfy: Offer의 계약")

**② `Dispatcher.Submit`의 계약.** Dispatcher는 이벤트를 받아 대기열에 넣는 부품이고, `Submit`이 그 메서드입니다. 대기열이 가득 찼으면 R2~R4에 따라 거부하거나, 가장 오래된 비중요 이벤트를 밀어내거나(`ACCEPTED_EVICTED`), 하나를 먼저 처리해 자리를 만듭니다(`ACCEPTED_DRAINED`). Dispatcher 증명은 대기열의 위 계약(①)만 믿고 R1~R4를 지킨다고 증명합니다. `Limit`은 받기로 한 최대 개수, `Dispatched`는 처리기로 넘긴 이벤트 목록입니다.

In [15]:
lab.show("formal/Dispatcher.dfy", 80, 106, title="formal/Dispatcher.dfy: Submit의 계약")

`Remove(s, k)`: s에서 k번째를 뺀 것. `IndexOfNonCritical(s)`: s에서 처음 나오는 비중요 이벤트의 위치. `AllCritical(s)`: s가 모두 중요 이벤트.

**해 볼 것** (답은 아래 셀에 적어 보세요)
1. ②에서 R3과 R4가 각각 어느 줄인지 찾습니다.
2. ①의 사후조건 중 "가득 찼을 때" 줄(75줄)을 지웠다고 합시다. 어떤 잘못된 구현이 검증을 통과하게 됩니까?
3. ②에서 `r == ACCEPTED <==> ...`처럼 "어느 결과가 나오는지"를 정하는 줄이 없다면, 두 구현이 같은 입력에 같은 결과를 낸다고 말할 수 있습니까?

(여기에 답을 적으세요)

<details><summary><b>답 보기</b></summary>

1. **R3**: 100~102줄(`ACCEPTED_EVICTED`이면 내용은 가장 오래된 비중요 이벤트를 뺀 나머지 + `e`). 98~99줄, 103~105줄도 남은 이벤트의 순서를 그대로 둔다고 말합니다. **R4**: 95~96줄(`REJECTED_FULL`은 가득 찼고, 비중요 이벤트이고, autoDrain이 꺼져 있을 때만).
2. 가득 찼을 때 FULL을 돌려주면서 **대기열 내용을 지우거나 바꾸는** 구현도 통과합니다. "가득 찼으면 내용 그대로"라고 말하는 줄이 없기 때문입니다.
3. 말할 수 없습니다. "결과가 X이면 상태는 Y"만 있으면, 같은 입력에 어떤 결과를 낼지는 구현마다 다를 수 있습니다. 89~96줄처럼 결과를 정하는 `<==>` 줄이 있어야 "이 계약을 지키는 구현은 모두 같은 입력에 같은 결과를 낸다"고 말할 수 있습니다.
</details>

## B-4. 차분 테스트 (10분)

Dafny 증명은 모델에 대한 것이고, 실제로 배포하는 것은 Java입니다. 차분 테스트는 **같은 연산을 Java와 모델 양쪽에 넣고 매 연산 뒤의 결과와 상태를 비교**합니다.

| 항목 | 값 |
|---|---|
| 반복 | 연산열 300개, 각 60개 연산. 무작위 시드 42 고정(매번 같은 입력) |
| 대기열 연산 비율 | offer 55%, poll 35%, peek 6%, clear 4%. offer의 20%는 중요 이벤트 |
| 대기열 크기 | 4칸 (작게 잡아야 가득 참·한 바퀴 돌기 같은 경계 상황이 자주 생깁니다) |
| 전력 값 | -5, 0, 1, 120, …, 4000, 4001, `Integer.MAX_VALUE` 같은 경계 값에서 고름 |
| 불일치가 나면 | 연산을 하나씩 빼 보며, 같은 불일치가 나는 가장 짧은 연산열로 줄여서 보여 줌 |

In [16]:
lab.run("difftest")

$ make difftest


[queue] 300 runs, 18000 ops: Java and model agree


[dispatcher] 300 runs, 18300 ops: Java and model agree


[power] 300 runs, 18000 ops: Java and model agree


**약점.** 무작위로 고른 입력만 봅니다. 큰 크기에서만 나는 문제나 경계 값 목록에 없는 숫자는 놓칠 수 있습니다. 차분 테스트 통과는 "시도한 입력에서 모델과 Java가 같았다"는 뜻이지 증명이 아닙니다.

### 해 볼 것: 일부러 버그를 넣고 차분 테스트가 잡는지 보기

`RingEventQueue.poll`에서 head를 옮기는 줄을 지웁니다.

In [17]:
Q = "legacy/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java"
lab.patch(Q, "        head = (head + 1) % buf.length;\n", "")

바꿨습니다: legacy/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java   (되돌리기: lab.restore('legacy/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java'))


In [18]:
lab.run("difftest", WHAT="queue")

$ make difftest WHAT=queue


[queue] MISMATCH (run 0, seed 42)


  minimal sequence (3 ops): [offer(1,0), offer(2,0), poll(0)]


  step 2 poll(0): contents java=[null] model=[2]


이벤트 2개를 넣고 하나를 꺼내면, Java는 head를 옮기지 않아 빈 칸(null)을 가리키고 모델은 [2]가 남습니다. 이 3단계짜리 연산열을 그대로 회귀 테스트로 옮길 수 있습니다.

**꼭 되돌립니다.**

In [19]:
lab.restore(Q)
lab.run("test")

되돌렸습니다: legacy/src/main/java/com/dxlab/homehub/queue/RingEventQueue.java
$ make test


tests: 38, passed: 38, failed: 0
